# Earnings Call Transcript Predictor
### Predicting Stock Returns & Sentiment from Earnings Call Transcripts using FinBERT
**Shannon Maccallum**

**Pipeline:**
1. Load and preprocess earnings call transcripts
2. Extract FinBERT sentiment scores
3. Fine-tune FinBERT to predict post-earnings returns
4. Generate Buy/Sell/Hold signals
5. Visualize results

## 1. Setup & Imports

In [ ]:
import json
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoTokenizer, AutoModel, pipeline
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Load Data

In [ ]:
data_dir = Path('/gpfs/home/smaccall/earnings-call-prediction/notebooks/data/transcripts')

def flatten_transcript(data):
    """Join speaker turns into single string."""
    if isinstance(data['transcript'], list):
        return ' '.join([turn['content'] for turn in data['transcript']])
    return data['transcript']

records = []
for path in data_dir.glob('*.json'):
    with open(path) as f:
        data = json.load(f)
    if 'transcript' in data and 'return_pct' in data:
        data['transcript'] = flatten_transcript(data)
        records.append(data)

print(f'Loaded {len(records)} transcripts')
print(f'Example: {records[0]["symbol"]} {records[0]["quarter"]} -> {records[0]["return_pct"]:.2f}%')

## 3. FinBERT Sentiment Analysis
FinBERT was pre-trained on financial text to classify sentiment. We extract sentiment scores for each transcript and correlate with actual returns.

In [ ]:
print('Loading FinBERT sentiment pipeline...')
sentiment_pipe = pipeline(
    'text-classification',
    model='ProsusAI/finbert',
    tokenizer='ProsusAI/finbert',
    device=0 if torch.cuda.is_available() else -1
)

def get_sentiment(text, chunk_size=400):
    """Score transcript sentiment. Uses first chunk to stay within token limit."""
    result = sentiment_pipe(text[:chunk_size], truncation=True)[0]
    label, score = result['label'], result['score']
    if label == 'positive': return score
    elif label == 'negative': return -score
    return 0.0

print('Extracting sentiment scores...')
for r in records:
    r['sentiment_score'] = get_sentiment(r['transcript'])

print(f'Done. Sample sentiment: {records[0]["sentiment_score"]:.3f}')

## 4. Sentiment vs Returns Analysis

In [ ]:
sentiments = [r['sentiment_score'] for r in records]
returns = [r['return_pct'] for r in records]
symbols = [r['symbol'] for r in records]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: sentiment vs return
colors = ['green' if r > 0 else 'red' for r in returns]
axes[0].scatter(sentiments, returns, c=colors, alpha=0.6, s=60, edgecolors='black', linewidth=0.3)
axes[0].axhline(0, color='black', linewidth=0.8, alpha=0.4)
axes[0].axvline(0, color='black', linewidth=0.8, alpha=0.4)
# Add correlation line
z = np.polyfit(sentiments, returns, 1)
p = np.poly1d(z)
x_line = np.linspace(min(sentiments), max(sentiments), 100)
axes[0].plot(x_line, p(x_line), 'b--', alpha=0.7, label=f'Trend')
corr = np.corrcoef(sentiments, returns)[0,1]
axes[0].set_xlabel('FinBERT Sentiment Score', fontsize=11)
axes[0].set_ylabel('3-Day Post-Earnings Return (%)', fontsize=11)
axes[0].set_title(f'Transcript Sentiment vs Stock Return\n(Pearson r = {corr:.3f})', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Distribution of sentiment by return direction
pos_sent = [s for s, r in zip(sentiments, returns) if r > 0]
neg_sent = [s for s, r in zip(sentiments, returns) if r <= 0]
axes[1].hist(pos_sent, bins=20, alpha=0.6, color='green', label=f'Positive return (n={len(pos_sent)})', density=True)
axes[1].hist(neg_sent, bins=20, alpha=0.6, color='red', label=f'Negative return (n={len(neg_sent)})', density=True)
axes[1].axvline(np.mean(pos_sent), color='green', linestyle='--', linewidth=2)
axes[1].axvline(np.mean(neg_sent), color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Sentiment Score', fontsize=11)
axes[1].set_ylabel('Density', fontsize=11)
axes[1].set_title('Sentiment Distribution by Return Direction', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('sentiment_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Correlation between sentiment and returns: {corr:.3f}')
print(f'Mean sentiment (positive returns): {np.mean(pos_sent):.3f}')
print(f'Mean sentiment (negative returns): {np.mean(neg_sent):.3f}')

## 5. Dataset & Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')

class EarningsDataset(Dataset):
    def __init__(self, records, tokenizer, max_length=512):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.records)
    
    def __getitem__(self, idx):
        record = self.records[idx]
        encoding = self.tokenizer(
            record['transcript'],
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(record['return_pct'], dtype=torch.float),
            'sentiment': torch.tensor(record['sentiment_score'], dtype=torch.float),
            'symbol': record.get('symbol', ''),
            'quarter': record.get('quarter', ''),
        }

dataset = EarningsDataset(records, tokenizer)
print(f'Dataset size: {len(dataset)}')

In [ ]:
class EarningsModel(nn.Module):
    """
    FinBERT fine-tuned for return regression.
    Takes transcript text -> predicts post-earnings return %
    """
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained('ProsusAI/finbert')
        self.dropout = nn.Dropout(0.1)
        self.regressor = nn.Linear(768, 1)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = outputs.last_hidden_state[:, 0, :]
        cls = self.dropout(cls)
        return self.regressor(cls).squeeze(-1)

model = EarningsModel().to(device)
print(f'Model loaded on {device}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 6. Train/Test Split & Training

In [ ]:
# 80/20 split
n_test = max(1, int(len(dataset) * 0.2))
n_train = len(dataset) - n_test
train_ds, test_ds = random_split(dataset, [n_train, n_test])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=8)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
criterion = nn.MSELoss()

print(f'Train: {n_train} | Test: {n_test}')

In [ ]:
train_losses = []
val_rmses = []
dir_accs = []

EPOCHS = 5

for epoch in range(EPOCHS):
    # Train
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        optimizer.zero_grad()
        preds = model(input_ids, attention_mask)
        loss = criterion(preds, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
    
    # Eval
    model.eval()
    val_loss = 0
    correct = 0
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            preds = model(input_ids, attention_mask)
            val_loss += criterion(preds, labels).item()
            correct += ((preds >= 0) == (labels >= 0)).sum().item()
    
    train_losses.append(total_loss / len(train_loader))
    val_rmses.append((val_loss / len(test_loader)) ** 0.5)
    dir_accs.append(correct / n_test)
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {train_losses[-1]:.4f} | RMSE: {val_rmses[-1]:.4f} | Dir Acc: {dir_accs[-1]:.2f}')

torch.save(model.state_dict(), 'best_model.pt')
print('Model saved!')

## 7. Training Curves

In [ ]:
epochs = list(range(1, EPOCHS + 1))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, train_losses, 'b-o', linewidth=2, markersize=6)
axes[0].set_title('Training Loss (MSE)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, val_rmses, 'r-o', linewidth=2, markersize=6)
axes[1].set_title('Validation RMSE', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('RMSE (%)')
axes[1].grid(alpha=0.3)

axes[2].plot(epochs, dir_accs, 'g-o', linewidth=2, markersize=6)
axes[2].axhline(0.5, color='gray', linestyle='--', alpha=0.7, label='Random baseline (0.5)')
axes[2].set_title('Directional Accuracy', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy')
axes[2].legend(fontsize=9)
axes[2].set_ylim(0, 1)
axes[2].grid(alpha=0.3)

plt.suptitle('FinBERT Training Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Predictions & Buy/Sell/Hold Signals

In [ ]:
model.eval()
all_preds = []
all_labels = []
all_sentiments = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        preds = model(input_ids, attention_mask)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch['label'].numpy())
        all_sentiments.extend(batch['sentiment'].numpy())

THRESHOLD = 2.0

def signal(return_pct, threshold=THRESHOLD):
    if return_pct > threshold: return 'BUY'
    elif return_pct < -threshold: return 'SELL'
    return 'HOLD'

print(f'{'Predicted':>10} {'Actual':>10} {'Sentiment':>10} {'Signal':>8} {'Correct?':>10}')
print('-' * 55)
correct = 0
for pred, actual, sent in zip(all_preds, all_labels, all_sentiments):
    pred_sig = signal(pred)
    actual_sig = signal(actual)
    match = '✓' if pred_sig == actual_sig else '✗'
    if pred_sig == actual_sig: correct += 1
    print(f'{pred:>10.2f}% {actual:>10.2f}% {sent:>10.3f} {pred_sig:>8} {match:>10}')

print(f'\nSignal Accuracy (±{THRESHOLD}%): {correct}/{len(all_preds)} = {correct/len(all_preds):.2f}')

## 9. Prediction Visualizations

In [ ]:
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
errors = all_preds - all_labels

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Predicted vs Actual
axes[0].scatter(all_labels, all_preds, alpha=0.7, color='steelblue', s=80, edgecolors='black', linewidth=0.5)
axes[0].plot([-10, 10], [-10, 10], 'r--', alpha=0.5, label='Perfect prediction')
axes[0].axhline(0, color='black', linewidth=0.5, alpha=0.3)
axes[0].axvline(0, color='black', linewidth=0.5, alpha=0.3)
axes[0].set_xlabel('Actual Return (%)', fontsize=11)
axes[0].set_ylabel('Predicted Return (%)', fontsize=11)
axes[0].set_title('Predicted vs Actual Returns', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: Side by side bar
x = np.arange(len(all_labels))
width = 0.35
axes[1].bar(x - width/2, all_labels, width, label='Actual', color='steelblue', alpha=0.7)
axes[1].bar(x + width/2, all_preds, width, label='Predicted', color='orange', alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Test Sample', fontsize=11)
axes[1].set_ylabel('Return (%)', fontsize=11)
axes[1].set_title('Actual vs Predicted Per Sample', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

# Plot 3: Error distribution
axes[2].hist(errors, bins=15, color='steelblue', alpha=0.7, edgecolor='black')
axes[2].axvline(0, color='red', linestyle='--', label='Zero error')
axes[2].axvline(np.mean(errors), color='orange', linestyle='--', label=f'Mean: {np.mean(errors):.2f}%')
axes[2].set_xlabel('Prediction Error (%)', fontsize=11)
axes[2].set_ylabel('Count', fontsize=11)
axes[2].set_title('Distribution of Prediction Errors', fontsize=12, fontweight='bold')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('prediction_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'MAE:  {np.mean(np.abs(errors)):.2f}%')
print(f'RMSE: {np.sqrt(np.mean(errors**2)):.2f}%')
print(f'Bias: {np.mean(errors):.2f}%')

## 10. Signal Accuracy by Threshold

In [ ]:
thresholds = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
accuracies = []
for t in thresholds:
    correct = sum(1 for p, a in zip(all_preds, all_labels)
                  if signal(p, t) == signal(a, t))
    accuracies.append(correct / len(all_preds))
    print(f'Threshold ±{t}%: {correct}/{len(all_preds)} = {correct/len(all_preds):.2f}')

plt.figure(figsize=(8, 5))
plt.bar(thresholds, accuracies, color='steelblue', alpha=0.7, width=0.3, edgecolor='black')
plt.axhline(0.5, color='red', linestyle='--', linewidth=2, label='Random baseline (0.50)')
plt.xlabel('Buy/Sell Threshold (%)', fontsize=11)
plt.ylabel('Signal Accuracy', fontsize=11)
plt.title('Buy/Sell/Hold Signal Accuracy by Threshold', fontsize=12, fontweight='bold')
plt.legend(fontsize=10)
plt.ylim(0, 1)
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('signal_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Summary

In [ ]:
print('='*50)
print('MODEL SUMMARY')
print('='*50)
print(f'Training samples:     {n_train}')
print(f'Test samples:         {n_test}')
print(f'Final Train Loss:     {train_losses[-1]:.4f}')
print(f'Final Val RMSE:       {val_rmses[-1]:.4f}%')
print(f'Final Dir Accuracy:   {dir_accs[-1]:.2f}')
print(f'MAE:                  {np.mean(np.abs(errors)):.2f}%')
print(f'Sentiment-Return Corr:{np.corrcoef(sentiments, returns)[0,1]:.3f}')
print(f'Best Signal Accuracy: {max(accuracies):.2f} at ±{thresholds[np.argmax(accuracies)]}%')
print('='*50)